# Comparing IRL Estimators: GCL vs NFXP vs MaxEnt

This notebook compares three approaches to learning from demonstrated behavior:

1. **NFXP** (Rust 1987) - Nested Fixed Point structural estimation
2. **MaxEnt IRL** (Ziebart 2008) - Maximum Entropy Inverse Reinforcement Learning  
3. **GCL** (Finn et al. 2016) - Guided Cost Learning with neural networks

We use the classic Rust bus engine replacement data as our test case.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from econirl.datasets import load_rust_bus
from econirl.estimators import NFXP, MaxEntIRL, GCL

# Set random seed for reproducibility
np.random.seed(42)

## Load Data

The Rust (1987) dataset contains monthly observations of bus mileage and engine replacement decisions.

In [ ]:
# Load the full dataset
df = load_rust_bus()

print(f"Total observations: {len(df):,}")
print(f"Number of buses: {df['bus_id'].nunique()}")
print(f"Replacement rate: {df['replaced'].mean():.2%}")
print()
df.head(10)

In [ ]:
# Use a subset of buses for faster comparison
buses = df['bus_id'].unique()[:20]
df_subset = df[df['bus_id'].isin(buses)].copy()

print(f"Subset observations: {len(df_subset):,}")
print(f"Subset buses: {df_subset['bus_id'].nunique()}")

## 1. NFXP Estimation

NFXP (Nested Fixed Point) estimates structural parameters by maximizing the likelihood of observed choices, solving the dynamic programming problem at each evaluation.

In [ ]:
%%time

nfxp = NFXP(
    n_states=90,
    discount=0.99,
    verbose=True
)

nfxp.fit(
    data=df_subset,
    state='mileage_bin',
    action='replaced',
    id='bus_id'
)

print(nfxp.summary())

## 2. MaxEnt IRL Estimation

Maximum Entropy IRL (Ziebart 2008) recovers a reward function that makes the demonstrated behavior most likely under a maximum entropy policy.

In [ ]:
# Create state features for MaxEnt IRL
n_states = 90
s = np.arange(n_states)
features = np.column_stack([
    s / 100,           # Linear mileage cost
    (s / 100) ** 2     # Quadratic mileage cost
])

print(f"Feature matrix shape: {features.shape}")
print(f"Features for state 0: {features[0]}")
print(f"Features for state 45: {features[45]}")
print(f"Features for state 89: {features[89]}")

In [ ]:
%%time

maxent = MaxEntIRL(
    n_states=90,
    n_actions=2,
    discount=0.99,
    feature_matrix=features,
    feature_names=['linear', 'quadratic'],
    verbose=True
)

maxent.fit(
    data=df_subset,
    state='mileage_bin',
    action='replaced',
    id='bus_id'
)

print(maxent.summary())

## 3. GCL Estimation

Guided Cost Learning (Finn et al. 2016) uses a neural network to parameterize the cost function, learning directly from trajectory-level comparisons via importance sampling.

In [ ]:
%%time

gcl = GCL(
    n_states=90,
    n_actions=2,
    discount=0.99,
    embed_dim=32,
    hidden_dims=[64, 64],
    cost_lr=1e-3,
    max_iterations=100,
    n_sample_trajectories=50,
    trajectory_length=30,
    verbose=True
)

gcl.fit(
    data=df_subset,
    state='mileage_bin',
    action='replaced',
    id='bus_id'
)

print(gcl.summary())

## Compare Learned Policies

All three methods produce policies π(a|s). Let's compare the replacement probability as a function of mileage.

In [ ]:
states = np.arange(90)

# Get replacement probabilities (action=1)
nfxp_replace_prob = nfxp.policy_[:, 1]
maxent_replace_prob = maxent.policy_[:, 1]
gcl_replace_prob = gcl.policy_[:, 1]

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(states, nfxp_replace_prob, 'b-', label='NFXP', linewidth=2)
ax.plot(states, maxent_replace_prob, 'r--', label='MaxEnt IRL', linewidth=2)
ax.plot(states, gcl_replace_prob, 'g-.', label='GCL', linewidth=2)

ax.set_xlabel('Mileage Bin (5,000 miles each)', fontsize=12)
ax.set_ylabel('P(Replace Engine)', fontsize=12)
ax.set_title('Comparison of Learned Replacement Policies', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 89)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Compare Value Functions

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(states, nfxp.value_function_, 'b-', label='NFXP', linewidth=2)
ax.plot(states, maxent.value_function_, 'r--', label='MaxEnt IRL', linewidth=2)
ax.plot(states, gcl.value_function_, 'g-.', label='GCL', linewidth=2)

ax.set_xlabel('Mileage Bin', fontsize=12)
ax.set_ylabel('Value Function V(s)', fontsize=12)
ax.set_title('Comparison of Value Functions', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Compare Reward/Cost Structures

NFXP and MaxEnt use linear reward functions, while GCL learns a flexible neural network cost.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# NFXP: Structural parameters define linear cost
nfxp_cost_keep = nfxp.params_['theta_c'] * states  # Operating cost
nfxp_cost_replace = np.full(90, nfxp.params_['RC'])  # Replacement cost

axes[0].plot(states, nfxp_cost_keep, 'b-', label='Keep (operating cost)', linewidth=2)
axes[0].axhline(nfxp.params_['RC'], color='r', linestyle='--', label=f'Replace (RC={nfxp.params_["RC"]:.2f})')
axes[0].set_xlabel('Mileage Bin')
axes[0].set_ylabel('Cost')
axes[0].set_title('NFXP: Structural Costs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MaxEnt IRL: Linear reward from features
axes[1].plot(states, maxent.reward_, 'r-', linewidth=2)
axes[1].set_xlabel('Mileage Bin')
axes[1].set_ylabel('Reward R(s)')
axes[1].set_title('MaxEnt IRL: Learned Reward')
axes[1].grid(True, alpha=0.3)

# GCL: Neural network cost
gcl_cost_keep = gcl.cost_matrix_[:, 0]
gcl_cost_replace = gcl.cost_matrix_[:, 1]

axes[2].plot(states, gcl_cost_keep, 'g-', label='Keep (a=0)', linewidth=2)
axes[2].plot(states, gcl_cost_replace, 'm--', label='Replace (a=1)', linewidth=2)
axes[2].set_xlabel('Mileage Bin')
axes[2].set_ylabel('Cost c(s,a)')
axes[2].set_title('GCL: Neural Network Costs')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Method': ['NFXP', 'MaxEnt IRL', 'GCL'],
    'Log-Likelihood': [
        nfxp.log_likelihood_,
        maxent.log_likelihood_,
        gcl.log_likelihood_
    ],
    'Converged': [
        nfxp.converged_,
        maxent.converged_,
        gcl.converged_
    ],
    'Mean P(Replace)': [
        nfxp.policy_[:, 1].mean(),
        maxent.policy_[:, 1].mean(),
        gcl.policy_[:, 1].mean()
    ],
    'P(Replace) at s=0': [
        nfxp.policy_[0, 1],
        maxent.policy_[0, 1],
        gcl.policy_[0, 1]
    ],
    'P(Replace) at s=45': [
        nfxp.policy_[45, 1],
        maxent.policy_[45, 1],
        gcl.policy_[45, 1]
    ],
    'P(Replace) at s=89': [
        nfxp.policy_[89, 1],
        maxent.policy_[89, 1],
        gcl.policy_[89, 1]
    ],
})

comparison.set_index('Method', inplace=True)
print(comparison.to_string())

## Prediction Comparison

Compare how well each method predicts the actual replacement decisions in the data.

In [ ]:
# Get actual states and actions from the data
states_data = df_subset['mileage_bin'].values
actions_data = df_subset['replaced'].values

# Compute prediction accuracy for each method
def compute_accuracy(policy, states, actions):
    """Fraction of correctly predicted modal actions."""
    predicted = policy[states].argmax(axis=1)
    return (predicted == actions).mean()

def compute_log_likelihood(policy, states, actions):
    """Log-likelihood of observed actions."""
    probs = policy[states, actions]
    return np.log(probs + 1e-10).sum()

print("Prediction Accuracy (modal action):")
print(f"  NFXP:      {compute_accuracy(nfxp.policy_, states_data, actions_data):.2%}")
print(f"  MaxEnt:    {compute_accuracy(maxent.policy_, states_data, actions_data):.2%}")
print(f"  GCL:       {compute_accuracy(gcl.policy_, states_data, actions_data):.2%}")

print("\nOut-of-sample Log-Likelihood:")
print(f"  NFXP:      {compute_log_likelihood(nfxp.policy_, states_data, actions_data):.2f}")
print(f"  MaxEnt:    {compute_log_likelihood(maxent.policy_, states_data, actions_data):.2f}")
print(f"  GCL:       {compute_log_likelihood(gcl.policy_, states_data, actions_data):.2f}")

## Key Takeaways

| Method | Strengths | Limitations |
|--------|-----------|-------------|
| **NFXP** | Interpretable structural parameters; Asymptotic inference | Requires correct model specification |
| **MaxEnt IRL** | Feature-based reward; Fast convergence | Linear reward assumption |
| **GCL** | Flexible neural network cost; No linearity assumption | Harder to interpret; More hyperparameters |

All three methods learn policies that capture the key pattern: **replacement probability increases with mileage**.